In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [2]:
# Q4.1: PHÁT HIỆN CÁC NGÀY CÓ DẤU HIỆU QUÁ NHIỆT
q4_1_overheat = spark.sql('''
    SELECT
        date,
        COUNT(*) AS tong_ban_ghi,
        SUM(
            CASE
                WHEN Oil_temperature >= 75
                THEN 1 ELSE 0
            END) AS so_lan_vuot_75C,
        SUM(
            CASE
                WHEN Oil_temperature >= 80
                THEN 1 ELSE 0
            END) AS so_lan_vuot_80C,
        ROUND(MAX(Oil_temperature), 2) AS nhiet_do_cao_nhat,
        ROUND(AVG(Oil_temperature), 2) AS nhiet_do_trung_binh
    FROM sensor
    WHERE COMP = 0
    GROUP BY date
    HAVING so_lan_vuot_75C > 0
    ORDER BY so_lan_vuot_75C DESC
    LIMIT 10''')
print("\n========== EXECUTION PLAN ==========")
q4_1_overheat.explain(True)
print("\n--- TOP 10 NGÀY CÓ DẤU HIỆU QUÁ NHIỆT ---")
df_q4_1 = q4_1_overheat.toPandas()
try: display(df_q4_1)
except NameError: print(df_q4_1.to_string(index=False))


========== EXECUTION PLAN ==========
== Parsed Logical Plan ==
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['so_lan_vuot_75C DESC NULLS LAST], true
      +- 'UnresolvedHaving ('so_lan_vuot_75C > 0)
         +- 'Aggregate ['date], ['date, 'COUNT(1) AS tong_ban_ghi#711, 'SUM(CASE WHEN ('Oil_temperature >= 75) THEN 1 ELSE 0 END) AS so_lan_vuot_75C#712, 'SUM(CASE WHEN ('Oil_temperature >= 80) THEN 1 ELSE 0 END) AS so_lan_vuot_80C#713, 'ROUND('MAX('Oil_temperature), 2) AS nhiet_do_cao_nhat#714, 'ROUND('AVG('Oil_temperature), 2) AS nhiet_do_trung_binh#715]
            +- 'Filter ('COMP = 0)
               +- 'UnresolvedRelation [sensor], [], false

== Analyzed Logical Plan ==
date: date, tong_ban_ghi: bigint, so_lan_vuot_75C: bigint, so_lan_vuot_80C: bigint, nhiet_do_cao_nhat: double, nhiet_do_trung_binh: double
GlobalLimit 10
+- LocalLimit 10
   +- Sort [so_lan_vuot_75C#712L DESC NULLS LAST], true
      +- Filter (so_lan_vuot_75C#712L > cast(0 as bigint))
         +- Aggregate [date#24],

,date,tong_ban_ghi,so_lan_vuot_75C,so_lan_vuot_80C,nhiet_do_cao_nhat,nhiet_do_trung_binh
0,2020-06-06,7343,5930,0,78.03,75.40
1,2020-06-05,5546,4669,0,77.65,74.78
2,2020-03-29,7199,4287,2169,83.13,77.24
3,2020-06-07,4888,3828,0,77.08,75.46
4,2020-03-12,4765,3159,0,78.03,74.31
5,2020-04-18,8592,2561,0,78.17,74.33
6,2020-07-15,4166,2224,1508,89.05,77.41
7,2020-05-30,2901,2159,0,76.65,73.28
8,2020-03-27,2468,1108,0,77.83,71.55
9,2020-04-12,4553,763,0,76.30,73.17
